In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
from pathlib import Path
import pandas as pd
import matplotlib
import os
import numpy as np
import scanpy as sc
from ipywidgets import Widget
import geopandas as gpd
print(dega.__version__)

/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWar

0.13.0


In [3]:
Widget.close_all()

## Inputs

In [4]:
technology = "Xenium"
sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
data_dir = f"data/xenium_data/{sample}"
segmentation_suffix = ""
path_landscape_files = f'data/landscape_files/{sample}_09-29-25'
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

## Make AnnData

In [5]:
# Load h5ad file, if exists already
adata = sc.read_h5ad(f'{path_landscape_files}/{Path(path_landscape_files).name}.h5ad')

/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [6]:
# adata = dega.pre._make_xenium_anndata(data_dir, path_landscape_files, write=True)

## Add default leiden clustering to the Anndata

In [7]:
meta_cell = pd.read_parquet(f"{path_landscape_files}/cell_metadata{segmentation_suffix}.parquet")
default_clustering, _clusters, _ser_counts = dega.pre._load_xenium_cluster_data(data_dir, meta_cell)

cluster_to_cell_id = default_clustering["cluster"].to_dict()
adata.obs["leiden"] = adata.obs["cell_id"].map(cluster_to_cell_id)

## Rank and save marker genes

In [8]:
n_genes = 100

In [9]:
# # Run ranking
# sc.tl.rank_genes_groups(adata, groupby="leiden", method="t-test", use_raw=False, show_progress=True)

# # Save markers
# marker_df = dega.qc._get_ranked_genes_df(adata, n_genes)
# marker_df.to_csv(f"{path_landscape_files}/marker_genes_by_cluster-{n_genes}_genes.csv", index=False)

### Next Steps:

#### 1. Uploaded "marker_genes_by_cluster-{n_genes}_genes.csv" on ChatGPT, and asked for tentative cell types.
#### 2. "Predicted_Cell_Types_Def_Clustering-{n_genes}_genes.csv" has the predicted cell types for each cluster based on the top 10 marker genes, with the cluster number included in the label.

In [10]:
# cell types

pred_cell_types_df = pd.read_csv(f"{path_landscape_files}/Predicted_Cell_Types_Def_Clustering-{n_genes}_genes.csv")

pred_cell_types_df.drop(['Unnamed: 0'], axis=1, inplace=True)
pred_cell_types_df["category"] = pred_cell_types_df["predicted_cell_type"].str.split("_").str[0]
pred_cell_types_df['cluster'] = pred_cell_types_df['cluster'].astype('string')
pred_cell_types_df.set_index('cluster', inplace=True)

cluster_to_celltype = pred_cell_types_df["predicted_cell_type"].to_dict()

pred_cell_types_df.reset_index(inplace=True)
pred_cell_types_df.head()

,cluster,predicted_cell_type,category
0,1,Unknown_1,Unknown
1,2,Fibroblast_2,Fibroblast
2,3,Smooth muscle_3,Smooth muscle
3,4,Unknown_4,Unknown
4,5,Macrophage_5,Macrophage


## Read cluster related files generated by Celldega 

In [11]:
cluster = pd.read_parquet(f"{path_landscape_files}/cell_clusters/cluster.parquet")
meta_cluster = pd.read_parquet(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet")
df_sig = pd.read_parquet(f"{path_landscape_files}/df_sig.parquet")

## Modify cluster related files generated by Celldega, to include cell types

In [12]:
# cluster["predicted_cell_type"] = cluster["cluster"].map(cluster_to_celltype)
# cluster.fillna('None')
# cluster.drop(['cluster'], axis=1, inplace=True)
# cluster.rename(columns={'predicted_cell_type':'cluster'}, inplace=True)

# os.rename(f"{path_landscape_files}/cell_clusters/cluster.parquet", 
#           f"{path_landscape_files}/cell_clusters/cluster_default.parquet")

# cluster.to_parquet(f"{path_landscape_files}/cell_clusters/cluster.parquet")

In [13]:
# meta_cluster.index = meta_cluster.index.astype('string')
# meta_cluster.index = meta_cluster.index.map(cluster_to_celltype)

# os.rename(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet", 
#           f"{path_landscape_files}/cell_clusters/meta_cluster_default.parquet")

# meta_cluster.to_parquet(f"{path_landscape_files}/cell_clusters/meta_cluster.parquet")

In [14]:
# df_sig.columns = df_sig.columns.astype('string')
# df_sig.columns = df_sig.columns.map(cluster_to_celltype)

# os.rename(f"{path_landscape_files}/df_sig.parquet", 
#           f"{path_landscape_files}/df_sig_default.parquet")

# df_sig.to_parquet(f"{path_landscape_files}/df_sig.parquet")

### Make hextiles

In [15]:
adata.obs['leiden'] = adata.obs['cell_id'].map(cluster['cluster'])

In [16]:
data = dega.nbhd._get_gdf_cell(adata)
gdf_nbhd = dega.nbhd.generate_hex_grid(data, diameter=150)

## Trx-unassignment-proportion per Hextile using NBHD module methods

In [17]:
gdf_trx = dega.nbhd._get_gdf_trx(data_dir)

In [ ]:
nbhd_meta = dega.nbhd.get_nbhd_meta(gdf_nbhd = gdf_nbhd,
                                    unique_nbhd_col="name",
                                    gdf_trx = gdf_trx,
                                    gdf_cell = data)

nbhd_meta.head()

Calculating NBM


/Users/jishar/Documents/celldega/src/celldega/nbhd/neighborhoods.py:340: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  summary["area_squm"] = gdf_nbhd.geometry.area.round(2)
/Users/jishar/Documents/celldega/src/celldega/nbhd/neighborhoods.py:341: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  summary["perimeter_um"] = gdf_nbhd.geometry.length.round(2)


In [ ]:
nbhd_meta['unassigned_trx_prop'] = np.where(
    nbhd_meta['total_trx'] > 0,
    nbhd_meta['unassigned_trx_count'] / nbhd_meta['total_trx'],
    np.nan
)

nbhd_meta['unassigned_trx_prop'].fillna(0, inplace=True)
nbhd_meta.head()

In [ ]:
gdf_nbhd['unassigned_trx_count'] = gdf_nbhd['name'].map(nbhd_meta['unassigned_trx_count'])
gdf_nbhd.head()

## Cell-cluster by Hextile using NBHD module methods

In [ ]:
adata_nbp, gdf_nbhd = dega.nbhd.calc_nbp(data, gdf_nbhd, category="cluster")

In [20]:
# Clustering
sc.pp.normalize_total(adata_nbp, inplace=True)
sc.pp.log1p(adata_nbp)
sc.pp.neighbors(adata_nbp, n_neighbors=10)
sc.tl.leiden(adata_nbp, resolution=1)

/Users/jishar/Documents/celldega/dega/lib/python3.13/site-packages/scanpy/preprocessing/_normalization.py:234: UserWarning: Some cells have zero counts
  warn(UserWarning("Some cells have zero counts"))


In [21]:
population_distribution = pd.DataFrame(
    adata_nbp.X, index=adata_nbp.obs_names, columns=adata_nbp.var_names
)

In [22]:
# Add clustering and proportions to hex GeoDataFrame
gdf_nbhd = gdf_nbhd.set_index("name")
gdf_nbhd["leiden"] = gdf_nbhd.index.map(adata_nbp.obs['leiden'])
gdf_nbhd["niche"] = [f"{cluster}" for cluster in gdf_nbhd["leiden"].values]
gdf_nbhd = gdf_nbhd.join(population_distribution)
gdf_nbhd.reset_index(inplace=True)

In [23]:
# Dissolve to form niches
gdf_niche = dega.nbhd._dissolve_by_category(gdf_nbhd, "leiden")
gdf_niche["name"] = [f"{c}" for c in gdf_niche["leiden"]]
gdf_niche.head()

,leiden,geometry,name,niche,Endothelial_13,Endothelial_22,Epithelial_20,Epithelial_23,Epithelial_7,Fibroblast_15,...,Unknown_1,Unknown_10,Unknown_11,Unknown_12,Unknown_14,Unknown_17,Unknown_24,Unknown_25,Unknown_4,Unknown_8
0,0,"MULTIPOLYGON (((1247.05703 2646.39656, 1247.05...",0,0,0.000000,0.000000,0.000000,0.000000,0.206536,0.005865,...,0.142316,0.167506,0.011696,0.000000,0.167506,0.000000,0.000000,0.000000,0.073688,0.040351
1,1,"MULTIPOLYGON (((4819.41182 4783.89656, 4754.45...",1,1,0.169899,0.000000,0.000000,0.036368,0.000000,0.000000,...,0.000000,0.000000,0.154151,0.000000,0.000000,0.018349,0.018349,0.000000,0.000000,0.000000
2,2,"MULTIPOLYGON (((2546.09514 1071.39656, 2546.09...",2,2,0.017392,0.017392,0.017392,0.000000,0.017392,0.034486,...,0.338975,0.034486,0.000000,0.017392,0.000000,0.000000,0.000000,0.017392,0.084083,0.034486
3,3,"MULTIPOLYGON (((3260.56610 5233.89656, 3195.61...",3,3,0.082692,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.188052,0.000000,0.000000,0.033902,0.113944,0.000000,0.000000,0.000000
4,4,"MULTIPOLYGON (((467.63417 5571.39656, 402.6822...",4,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.344840,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Clustergram: cell_population-by-hextile_nbhd 

In [24]:
gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
gdf_nbhd_.set_index('name', inplace=True)
gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
gdf_nbhd_.head()

,Endothelial_13,Endothelial_22,Epithelial_20,Epithelial_23,Epithelial_7,Fibroblast_15,Fibroblast_19,Fibroblast_2,Fibroblast_9,Macrophage_18,...,Unknown_1,Unknown_10,Unknown_11,Unknown_12,Unknown_14,Unknown_17,Unknown_24,Unknown_25,Unknown_4,Unknown_8
name,,,,,,,,,,,,,,,,,,,,,
hex_25,0.017392,0.017392,0.017392,0.0,0.017392,0.034486,0.034486,0.191055,0.0,0.000000,...,0.338975,0.034486,0.000000,0.017392,0.000000,0.0,0.0,0.017392,0.084083,0.034486
hex_26,0.000000,0.000000,0.012903,0.0,0.000000,0.012903,0.000000,0.098846,0.0,0.025642,...,0.451985,0.000000,0.012903,0.000000,0.000000,0.0,0.0,0.012903,0.199489,0.000000
hex_27,0.013986,0.000000,0.000000,0.0,0.041385,0.027780,0.013986,0.131769,0.0,0.000000,...,0.372049,0.027780,0.013986,0.000000,0.013986,0.0,0.0,0.013986,0.168137,0.027780
hex_28,0.010257,0.010257,0.000000,0.0,0.020409,0.040410,0.000000,0.088728,0.0,0.020409,...,0.359763,0.010257,0.000000,0.000000,0.010257,0.0,0.0,0.000000,0.253603,0.030459
hex_29,0.000000,0.000000,0.000000,0.0,0.000000,0.021979,0.000000,0.218689,0.0,0.043485,...,0.105361,0.125163,0.064539,0.000000,0.000000,0.0,0.0,0.000000,0.144581,0.021979


In [25]:
gdf_nbhd_T = gdf_nbhd_.T
gdf_nbhd_T.head()

name,hex_25,hex_26,hex_27,hex_28,hex_29,hex_30,hex_31,hex_32,hex_33,hex_34,...,hex_5895,hex_5896,hex_5897,hex_5898,hex_5899,hex_5900,hex_5901,hex_5902,hex_5903,hex_5904
Endothelial_13,0.017392,0.000000,0.013986,0.010257,0.0,0.000000,0.035091,0.000000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.057158,0.169899,0.346276,0.000000,0.087011,0.071459,0.000000
Endothelial_22,0.017392,0.000000,0.000000,0.010257,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.057158,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Epithelial_20,0.017392,0.012903,0.000000,0.000000,0.0,0.017392,0.000000,0.010695,0.01005,0.021053,...,0.310155,0.0,0.189242,0.057158,0.036368,0.000000,0.223144,0.087011,0.071459,0.236389
Epithelial_23,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Epithelial_7,0.017392,0.000000,0.041385,0.020409,0.0,0.000000,0.174353,0.185899,0.11441,0.091434,...,0.167054,0.0,0.223144,0.000000,0.036368,0.000000,0.000000,0.000000,0.000000,0.064539


In [26]:
meta_col = pd.DataFrame(index=gdf_nbhd_T.columns.tolist())
top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index.tolist()
niches = gdf_nbhd.set_index("name").loc[top_cols, "niche"]
niches = pd.DataFrame(niches)
meta_col["niche"] = meta_col.index.map(niches["niche"])
meta_col[:5]

,niche
hex_25,2
hex_26,2
hex_27,2
hex_28,2
hex_29,9


In [27]:
meta_row = pd.DataFrame(index=gdf_nbhd_T.index.tolist())
top_rows = [row for row in gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index]
meta_row["category"] = meta_row.index

In [28]:
mat = dega.clust.Matrix(
    gdf_nbhd_T,
    name='parquet',
    meta_col=meta_col,
    col_attr=['niche'],
    meta_row=meta_row,
    row_entity="cell_cluster",
    col_entity="nbhd"
)

mat.downsample_to(axis='col', category='niche')
mat.norm(axis='row', by='zscore')
mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat, 
    width=500, 
    height=500,
)

## Visualize in Landscape view: Hextile NBHD

### Trx-assignment per Hextile

In [ ]:
gdf_nbhd_LF = gdf_nbhd.copy()
gdf_nbhd_LF.head()

In [ ]:
cmap = matplotlib.pyplot.get_cmap('Reds')

# Normalize values between 0 and 1
norm = (gdf_nbhd_LF['unassigned_trx_count'] - gdf_nbhd_LF['unassigned_trx_count'].min()) / (gdf_nbhd_LF['unassigned_trx_count'].max() - gdf_nbhd_LF['unassigned_trx_count'].min())

# Apply color map
gdf_nbhd_LF['color'] = norm.apply(cmap)
gdf_nbhd_LF['color'] = gdf_nbhd_LF['color'].apply(matplotlib.colors.to_hex)
gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area
gdf_nbhd_LF.head()

In [ ]:
# # save for future reference
# gdf_nbhd_LF.to_parquet(f"{path_landscape_files}/gdf_nbhd_meta.parquet")

### Hextile Niche

In [29]:
gdf_nbhd_LF = gdf_nbhd.copy()
gdf_nbhd_LF = gdf_nbhd_LF[['geometry','name','leiden']]
gdf_nbhd_LF.rename(columns={'leiden':'cat'}, inplace=True)
gdf_nbhd_LF.head()

,geometry,name,cat
0,"POLYGON ((6248.35374 58.89656, 6183.40184 96.3...",hex_25,2
1,"POLYGON ((6378.25755 58.89656, 6313.30565 96.3...",hex_26,2
2,"POLYGON ((6508.16136 58.89656, 6443.20946 96.3...",hex_27,2
3,"POLYGON ((6638.06517 58.89656, 6573.11327 96.3...",hex_28,2
4,"POLYGON ((6767.96898 58.89656, 6703.01708 96.3...",hex_29,9


In [30]:
categories = gdf_nbhd_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_nbhd_LF['color'] = gdf_nbhd_LF['cat'].astype(str).map(cat_to_hex)
gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area

/var/folders/_6/bhs42vt57t1dkb59k4sy0p440000gp/T/ipykernel_7346/1605102556.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_nbhd_LF['area'] = gdf_nbhd_LF['geometry'].area


In [31]:
# # save for future reference

# gdf_nbhd_LF.to_parquet(f"{path_landscape_files}/gdf_nbhd.parquet")
# gdf_nbhd_T.to_parquet(f"{path_landscape_files}/nbp.parquet")
# meta_col.to_parquet(f"{path_landscape_files}/meta_col.parquet")
# meta_row.to_parquet(f"{path_landscape_files}/meta_row.parquet")

## Trx-assignment per Hextile Landscape View 

In [ ]:
landscape_ist = dega.viz.Landscape(
    technology = technology,
    base_url = base_url,
    nbhd = gdf_nbhd_LF
)

In [ ]:
landscape_ist

## Cell-Type By Hextile-Niche Landscape-Clustergram View 

In [57]:
# dega.viz.landscape_clustergram(landscape_ist, cgm)